## Imports and config

### Make sure the paths in the config are correct

In [ ]:
import os
import os.path as osp
import sys
module_path = os.path.abspath(os.path.join('../..'))
sys.path.append(module_path)
from src.utils.db import DatabaseManager
from src.utils.process_config import load_config
from itables import show
import pandas as pd

import numpy as np

config_path = os.path.join(module_path, 'src', 'config', 'edit_192_external.json')

class Args:
    def __init__(self, config):
        self.config = config
args = Args(config=config_path)
config = load_config(args.config)
dbm = DatabaseManager(config)
print("Database manager initialized from config:", config_path)
dbm.print_db_schema_counts()





In [ ]:
# test rater - make sure to remove before calculating human ratings
exclude_raters = ["private.us-west-2.0fe3ef0902668b13"]

# columns to show in the leaderboard
score_columns = [
    "chamfer dist",
    "iou",
    "dino v2 sim",
    "validity",
    "claude instruction",
    "claude quality",
    "claude acceptance",
    "gemini instruction",
    "gemini quality",
    "gemini acceptance",
    "gpt instruction",
    "gpt quality",
    "gpt acceptance",
    "human instruction",
    "human quality",
    "human acceptance",
]

# users to show in the leaderboard
users = [
    "human requestor",
    "human baseline",
    "claude-sonnet-4.5_cadquery-script",
    "gemini-3-pro_cadquery-script",
    "gpt-5.2_cadquery-script",
]


rating_keys = []
user_name_mapping = {}
all_edits = {}
for edt in dbm.edits.find({}):
    all_edits[edt["_id"]] = {}

for rating in dbm.ratings.find({}):
    edit_id = rating["edit"]
    ignore_keys = ["_id", "edit", "user", 'result_info', 'job_info', 'submission_time', 'comments', 'flags']
    existing_edit = all_edits.get(edit_id, {})
    
    rating_user = rating["user"]

    existing_rating_user = existing_edit.get(rating_user, {})

    # only add keys that are not in ignore_keys
    for k, v in rating.items():
        if k not in ignore_keys:
            existing_rating_user[k] = v

    #update the edit's ratings with the new rating
    existing_edit[rating_user] = existing_rating_user
    all_edits[edit_id] = existing_edit

def process_single_rating(rating):
    # instruction and qualitiy human/vlm ratings have different defaults for doing nothing/failed runs according to the scoring rubric.
    
    new_rating = {}
    similarity_eval = rating.get("similarity_eval", {})
    # if similarity_eval is not None:
    new_rating["dino v2 sim"] = similarity_eval.get("dino similarity gt", 0.0)
    new_rating["chamfer dist"] = 1.0 / (similarity_eval.get("chamfer similarity gt", 9999) + 1e-8)
    new_rating["iou"] = similarity_eval.get("iou gt", 0.0)
    
    gemini_dict = rating.get('gemini-3-pro_30-720_edit-rating-6-img', {})
    new_rating["gemini instruction"] = (gemini_dict.get("score_instr", 2) - 1 ) / 6.0
    new_rating["gemini quality"] = (gemini_dict.get("score_quality", 1) - 1 ) / 6.0

    gpt_dict = rating.get('gpt-5.2_edit-rating-6-img', {})
    new_rating["gpt instruction"] = (gpt_dict.get("score_instr", 2) - 1 ) / 6.0
    new_rating["gpt quality"] = (gpt_dict.get("score_quality", 1) - 1 ) / 6.0

    claude_dict = rating.get('claude-sonnet-4.5_edit-rating-6-img', {})
    new_rating["claude instruction"] = (claude_dict.get("score_instr", 2) - 1 ) / 6.0
    new_rating["claude quality"] = (claude_dict.get("score_quality", 1) - 1 ) / 6.0

    instruction_threshold = (5-1) / 6.0
    quality_threshold = (5-1) / 6.0

    human_instruction = []
    human_quality = []

    for k, v in rating.items():
        if "private" in k and k not in exclude_raters:
            for pk, pv in v.items():
                instr=False
                quality=False
                if "instr" in pk:
                    # if isinstance(pv, (int, float)):
                    if pv is not None and pv == pv:

                        # score = pv if isinstance(pv, (int, float)) else 2.0
                        score = (pv - 1) / 6.0
                        human_instruction.append(score)
                        instr=True
                if "quality" in pk:
                    # if isinstance(pv, (int, float)):
                    if pv is not None and pv == pv:
                        # score = pv if isinstance(pv, (int, float)) else 1.0
                        score = (pv - 1) / 6.0
                        human_quality.append(score)
                        quality=True
    new_rating["human instruction"] = float(np.median(human_instruction)) if human_instruction else 1/6.0
    new_rating["human quality"] = float(np.median(human_quality)) if human_quality else 0/6.0
    new_rating["human acceptance"] = float(new_rating["human instruction"] >= instruction_threshold and new_rating["human quality"] >= quality_threshold)
    new_rating["gemini acceptance"] = float(new_rating["gemini instruction"] >= instruction_threshold and new_rating["gemini quality"] >= quality_threshold)
    new_rating["gpt acceptance"] = float(new_rating["gpt instruction"] >= instruction_threshold and new_rating["gpt quality"] >= quality_threshold)
    new_rating["claude acceptance"] = float(new_rating["claude instruction"] >= instruction_threshold and new_rating["claude quality"] >= quality_threshold)    
    return new_rating


for k, v in all_edits.items():
    new_rating = process_single_rating(v)
    new_rating["edit_id"] = k
    new_rating["request_id"] = dbm.edits.find_one({"_id": k})["request"]

    edit_user_id = dbm.edits.find_one({"_id": k})["user"]
    request_user_id = dbm.requests.find_one({"_id": new_rating["request_id"]})["user"]
    edit_user = dbm.users.find_one({"_id": edit_user_id})
    request_user = dbm.users.find_one({"_id": request_user_id})
    new_rating["modality"] = dbm.requests.find_one({"_id": new_rating["request_id"]}).get("modality")

    if edit_user["is_human"]:
        if edit_user_id == request_user_id:
            new_rating["edit_user"] = "human requestor"
        else:
            new_rating["edit_user"] = "human baseline"
    else:
        new_rating["edit_user"] = edit_user_id

    # validity entry - checks that there is a toprightiso, as that means the preprocessing ran OK
    brep_id = dbm.edits.find_one({"_id": k})["brep_end"]
    images = dbm.get_brep_images(brep_id, views=["toprightiso"], format=["jpg", "png"])
    if images is not None and len(images) > 0:
        new_rating["validity"] = 1
    else:
        new_rating["validity"] = 0
    
    difficulty = dbm.requests.find_one({"_id": new_rating["request_id"]}).get("difficulty")
    new_rating["difficulty"] = difficulty

    all_edits[k] = new_rating

# turn into dataframe, with edit_id as index
df = pd.DataFrame.from_dict(all_edits, orient='index')

# Filter to only the specified users
df_filtered = df[df["edit_user"].isin(users)].copy()
df_melted = df_filtered.melt(id_vars=["edit_user"], value_vars=score_columns, var_name="Metric", value_name="Score")
df_grouped = df_melted.groupby(["edit_user", "Metric"], as_index=False)["Score"].mean()

# Enforce the desired user order and score order
df_grouped["edit_user"] = pd.Categorical(df_grouped["edit_user"], categories=users, ordered=True)
df_grouped["Metric"] = pd.Categorical(df_grouped["Metric"], categories=score_columns, ordered=True)
df_grouped = df_grouped.sort_values(["edit_user", "Metric"])

df_pivot_scores = df_grouped.pivot(index="edit_user", columns="Metric", values="Score")
df_pivot_scores = df_pivot_scores.reindex(index=users, columns=score_columns)

# set all values to 2 decimal places
df_pivot_scores = df_pivot_scores.round(2)

# Print results as tab-separated table for pasting into spreadsheet
# Print header
print("\t".join(["edit_user"] + score_columns))
# Print rows
for user in users:
    row_values = [f"{df_pivot_scores.loc[user, col]:.4f}" for col in score_columns]
    print("\t".join([user] + row_values))

# print results in nice table using itables
show(df_pivot_scores)



